# NMF on ModulAir 00682

In [13]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [14]:
#importing data from Modulair MOD-00682
data = pd.read_csv('MOD-00682.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-29T19:23:31Z,576100512,2025-12-29T14:23:31Z,MOD-00682,82.0,9.5,17.845,2.898,0.659,0.067,0.099,...,31.895,-13.269,14319,14320,14321,14469.0,14494.0,14544.0,14519.0,5.81
2025-12-29T19:22:31Z,576100515,2025-12-29T14:22:31Z,MOD-00682,82.1,9.5,19.233,3.362,0.798,0.113,0.093,...,31.417,-12.529,14319,14320,14321,14469.0,14494.0,14544.0,14519.0,6.83
2025-12-29T19:21:31Z,576100514,2025-12-29T14:21:31Z,MOD-00682,82.4,9.5,19.714,3.435,0.946,0.105,0.115,...,31.168,-13.787,14319,14320,14321,14469.0,14494.0,14544.0,14519.0,5.18
2025-12-29T19:20:31Z,576100513,2025-12-29T14:20:31Z,MOD-00682,83.0,9.5,20.809,3.720,0.963,0.134,0.150,...,31.617,-13.597,14319,14320,14321,14469.0,14494.0,14544.0,14519.0,7.16
2025-12-29T19:19:31Z,576098624,2025-12-29T14:19:31Z,MOD-00682,83.5,9.5,22.045,4.229,1.090,0.171,0.086,...,32.069,-14.149,14319,14320,14321,14469.0,14494.0,14544.0,14519.0,5.14


In [15]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-29T19:23:31Z,2025-12-29T14:23:31Z,821.109,2.480,31.895,-13.269,17.845,2.898,0.659,0.067,0.099,0.060
2025-12-29T19:22:31Z,2025-12-29T14:22:31Z,834.585,2.476,31.417,-12.529,19.233,3.362,0.798,0.113,0.093,0.069
2025-12-29T19:21:31Z,2025-12-29T14:21:31Z,841.351,2.480,31.168,-13.787,19.714,3.435,0.946,0.105,0.115,0.050
2025-12-29T19:20:31Z,2025-12-29T14:20:31Z,854.061,2.764,31.617,-13.597,20.809,3.720,0.963,0.134,0.150,0.069
2025-12-29T19:19:31Z,2025-12-29T14:19:31Z,866.432,2.766,32.069,-14.149,22.045,4.229,1.090,0.171,0.086,0.080


In [16]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-29T14:23:31Z,821.109,2.480,31.895,-13.269,17.845,2.898,0.659,0.067,0.099,0.060
1,2025-12-29T14:22:31Z,834.585,2.476,31.417,-12.529,19.233,3.362,0.798,0.113,0.093,0.069
2,2025-12-29T14:21:31Z,841.351,2.480,31.168,-13.787,19.714,3.435,0.946,0.105,0.115,0.050
3,2025-12-29T14:20:31Z,854.061,2.764,31.617,-13.597,20.809,3.720,0.963,0.134,0.150,0.069
4,2025-12-29T14:19:31Z,866.432,2.766,32.069,-14.149,22.045,4.229,1.090,0.171,0.086,0.080


In [17]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-29 14:23:31,821.109,2.480,31.895,-13.269,17.845,2.898,0.659,0.067,0.099,0.060
1,2025-12-29 14:22:31,834.585,2.476,31.417,-12.529,19.233,3.362,0.798,0.113,0.093,0.069
2,2025-12-29 14:21:31,841.351,2.480,31.168,-13.787,19.714,3.435,0.946,0.105,0.115,0.050
3,2025-12-29 14:20:31,854.061,2.764,31.617,-13.597,20.809,3.720,0.963,0.134,0.150,0.069
4,2025-12-29 14:19:31,866.432,2.766,32.069,-14.149,22.045,4.229,1.090,0.171,0.086,0.080


In [18]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-03-31 20:00:00,869.26,37.01,13.00,2.00,25.40,2.89,1.34,0.47,0.67,0.59
1,2025-03-31 21:00:00,909.42,39.49,10.87,2.53,34.53,5.53,1.91,0.59,0.83,0.76
2,2025-03-31 22:00:00,1066.79,41.32,11.80,2.83,36.65,9.72,2.59,0.64,0.81,0.69
3,2025-03-31 23:00:00,828.75,31.62,25.25,2.78,22.15,5.70,1.46,0.33,0.36,0.27
4,2025-04-01 00:00:00,716.48,29.67,31.26,2.51,17.51,3.57,0.95,0.21,0.24,0.17
...,...,...,...,...,...,...,...,...,...,...,...
6485,2025-12-29 10:00:00,1348.07,36.59,-12.43,35.31,35.57,16.35,5.93,1.23,1.06,0.63
6486,2025-12-29 11:00:00,983.03,34.19,-13.04,4.08,34.66,10.73,3.44,0.64,0.42,0.18
6487,2025-12-29 12:00:00,974.54,35.18,-13.33,10.24,35.07,11.05,3.55,0.66,0.43,0.16
6488,2025-12-29 13:00:00,994.99,34.99,-13.70,10.35,39.25,12.09,3.73,0.68,0.45,0.17


In [19]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,869.26,37.01,13.00,2.00,25.40,2.89,1.34,0.47,0.67,0.59
2025-03-31 21:00:00,909.42,39.49,10.87,2.53,34.53,5.53,1.91,0.59,0.83,0.76
2025-03-31 22:00:00,1066.79,41.32,11.80,2.83,36.65,9.72,2.59,0.64,0.81,0.69
2025-03-31 23:00:00,828.75,31.62,25.25,2.78,22.15,5.70,1.46,0.33,0.36,0.27
2025-04-01 00:00:00,716.48,29.67,31.26,2.51,17.51,3.57,0.95,0.21,0.24,0.17


In [20]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [21]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [22]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.465949,0.625486,0.142622,0.045652,0.283419,0.085351,0.082972,0.121762,0.251880,0.327778
2025-03-31 21:00:00,0.487476,0.667399,0.119254,0.057749,0.385293,0.163320,0.118266,0.152850,0.312030,0.422222
2025-03-31 22:00:00,0.571831,0.698327,0.129457,0.064597,0.408949,0.287064,0.160372,0.165803,0.304511,0.383333
2025-03-31 23:00:00,0.444234,0.534392,0.277016,0.063456,0.247155,0.168340,0.090402,0.085492,0.135338,0.150000
2025-04-01 00:00:00,0.384054,0.501437,0.342951,0.057293,0.195380,0.105434,0.058824,0.054404,0.090226,0.094444


In [23]:
data.to_csv(r'MOD-00682_timeseries_hourly_scaled.csv')

In [24]:
# Check start date of df to make sure that it's loaded in correctly
start = data.index.min()

end = data.index.min()

print(start, end)

2025-03-31 20:00:00 2025-03-31 20:00:00


## Implementing NMF

In [25]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [26]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.500837,0.608692,0.137098,0.058621,0.225103,0.168479,0.134576,0.148368,0.241643,0.265306
2025-03-31 21:00:00,0.561827,0.631927,0.103254,0.062608,0.306195,0.226653,0.177994,0.194498,0.313406,0.342392
2025-03-31 22:00:00,0.630227,0.669901,0.113116,0.068639,0.377234,0.252223,0.187591,0.200167,0.319864,0.347799
2025-03-31 23:00:00,0.486750,0.513590,0.264086,0.059724,0.234037,0.125501,0.085456,0.088136,0.144072,0.156358
2025-04-01 00:00:00,0.438057,0.474796,0.327344,0.057746,0.168989,0.080003,0.052920,0.054357,0.092588,0.100985
...,...,...,...,...,...,...,...,...,...,...
2025-12-28 09:00:00,0.386599,0.589925,0.036359,0.043025,0.132025,0.046509,0.017081,0.010144,0.014905,0.017917
2025-12-28 10:00:00,0.414853,0.607073,0.038659,0.045433,0.161359,0.055811,0.019540,0.010605,0.014428,0.016699
2025-12-28 11:00:00,0.415838,0.605281,0.025677,0.044801,0.169886,0.057617,0.018860,0.008857,0.010807,0.012467


In [27]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.052428,0.022757,0.060371,0.137753
1,0.052253,0.017063,0.084240,0.180130
2,0.055833,0.018701,0.104139,0.182616
3,0.046327,0.044164,0.059636,0.077451
4,0.044237,0.054822,0.039559,0.046725
...,...,...,...,...
5760,0.058487,0.005768,0.036565,0.003883
5761,0.060222,0.006145,0.044828,0.003032
5762,0.060172,0.003960,0.047580,0.000897
5763,0.060614,0.002208,0.058106,0.000000


In [28]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [29]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,4.926672,10.025450,0.035625,0.594610,0.000000,0.000000,0.000000,0.006990,0.072630,0.170716
Feature 2,1.979836,0.055251,5.942262,0.407267,0.538332,0.000000,0.000000,0.018097,0.147627,0.137536
Feature 3,2.336728,0.027049,0.000000,0.154513,3.525751,1.197798,0.381120,0.156891,0.091654,0.000000
Feature 4,0.409530,0.582082,0.000000,0.064248,0.000000,0.698108,0.809905,1.002649,1.661974,1.838257


In [30]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-03-31 20:00:00,0.052428,0.022757,0.060371,0.137753
2025-03-31 21:00:00,0.052253,0.017063,0.084240,0.180130
2025-03-31 22:00:00,0.055833,0.018701,0.104139,0.182616
2025-03-31 23:00:00,0.046327,0.044164,0.059636,0.077451
2025-04-01 00:00:00,0.044237,0.054822,0.039559,0.046725
...,...,...,...,...
2025-12-28 09:00:00,0.058487,0.005768,0.036565,0.003883
2025-12-28 10:00:00,0.060222,0.006145,0.044828,0.003032
2025-12-28 11:00:00,0.060172,0.003960,0.047580,0.000897


In [31]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,4.926672,10.025450,0.035625,0.594610,0.000000,0.000000,0.000000,0.006990,0.072630,0.170716
Factor 2,1.979836,0.055251,5.942262,0.407267,0.538332,0.000000,0.000000,0.018097,0.147627,0.137536
Factor 3,2.336728,0.027049,0.000000,0.154513,3.525751,1.197798,0.381120,0.156891,0.091654,0.000000
Factor 4,0.409530,0.582082,0.000000,0.064248,0.000000,0.698108,0.809905,1.002649,1.661974,1.838257


In [32]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.592770,0.195745,0.175009,0.018237,0.018239
no,0.554070,0.311847,0.089623,0.022158,0.022302
no2,0.980288,0.004439,0.001646,0.021066,-0.007440
o3,0.007311,1.002057,0.000000,0.000000,-0.009368
bin0,0.000000,0.173552,0.861032,0.000000,-0.034583
bin1,0.000000,0.000000,0.846830,0.293467,-0.140297
bin2,0.000000,0.000000,0.476565,0.602168,-0.078733
bin3,0.014441,0.030724,0.201769,0.766708,-0.013642
bin4,0.083657,0.139728,0.065714,0.708524,0.002377
bin5,0.175231,0.116006,0.000000,0.698369,0.010394


In [33]:
results.to_csv(r'MOD-00682_4_factor_results.csv')
comp.to_csv(r'MOD-00682_4_factor_comp.csv')
res.to_csv(r'MOD-00682_4_factor_resid.csv')

In [34]:
# check how many rows of data you have
len(data)

5765